# Tableau Workbook Metadata Extraction Pipeline

Organized into executable steps for readability. **All original Python code is preserved verbatim**; execute code cells in their displayed order. The main pipeline, exception handler, and cleanup block intentionally remain together.

## TABLEAU WORKBOOK METADATA EXTRACTION PIPELINE

Execute this section in sequence.

In [ ]:
# ============================================================
# TABLEAU WORKBOOK METADATA EXTRACTION PIPELINE
# ============================================================
#
# Supports:
#   1. Tableau Cloud / Tableau Server REST API
#   2. Tableau Metadata API (GraphQL)
#   3. Local .twb / .twbx files
#
# Major safeguards:
#   - REST API pagination
#   - Exact workbook-name filtering
#   - Full nested project hierarchy resolution
#   - Deterministic workbook selection by REST LUID
#   - Metadata API queried by workbook LUID
#   - Correct Tableau Metadata API endpoint:
#         /api/metadata/graphql
#   - Robust sign-in / sign-out cleanup
#   - Original exception preserved if cleanup fails
#   - No PAT secrets written to outputs
#   - Flat CONFIG structure throughout
#   - Excel sheet-name sanitization
#   - Safe CSV / JSON / Excel output
#   - TWB/TWBX XML extraction
#   - Metadata API + TWB metadata combined\n#   - Immutable metadata snapshots and row-level change log\n#   - Run/version/timestamp columns on metadata output tables\n#   - PNG screenshots for each Tableau workbook view
#
# ============================================================




## Step 0. IMPORTS

Load Python standard-library and third-party dependencies required by the extractor.

In [ ]:
# ============================================================
# 0. IMPORTS
# ============================================================

import os
import re
import json
import time
import hashlib
import zipfile
import unicodedata
import traceback
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
from typing import Any, Dict, List, Optional, Tuple
from getpass import getpass

import requests
import pandas as pd
import numpy as np

from lxml import etree

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

import os
from getpass import getpass

os.environ["TABLEAU_PAT_NAME"] = input(
    "Enter your Tableau Personal Access Token name: "
).strip()

os.environ["TABLEAU_PAT_SECRET"] = getpass(
    "Enter your Tableau Personal Access Token secret: "
)

if not os.environ["TABLEAU_PAT_NAME"]:
    raise ValueError("PAT name cannot be empty.")

if not os.environ["TABLEAU_PAT_SECRET"]:
    raise ValueError("PAT secret cannot be empty.")

print("Tableau PAT credentials are set for this notebook session.")



## Step 1. CONFIGURATION

Configure Tableau connectivity, workbook selection, metadata extraction, versioning, screenshots, and output options.

In [ ]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

CONFIG = {
    # --------------------------------------------------------
    # INPUT MODE
    # --------------------------------------------------------
    "mode": "cloud",                 # "cloud" or "local"
    "local_workbook": r"",           # e.g. r"C:\Tableau\Sales Dashboard.twbx"

    # --------------------------------------------------------
    # TABLEAU CLOUD / SERVER
    # --------------------------------------------------------
    "tableau_server": "https://your-tableau-pod.online.tableau.com",
    "site_content_url": "demo-site",
    "api_version": "3.29",

    # --------------------------------------------------------
    # TARGET WORKBOOK
    # --------------------------------------------------------
    "workbook_name": "Retail Performance Demo",

    # Leave blank to discover deterministically by:
    #   workbook name + project hierarchy
    #
    # If supplied, this LUID takes precedence and is validated
    # against the configured project hierarchy.
    "workbook_luid": "",

    # --------------------------------------------------------
    # EXPECTED PROJECT HIERARCHY
    # --------------------------------------------------------
    "project_path": [
        "CDM",
        "02 - CDM Enterprise Reports",
        "Commercial Finance",
    ],

    # --------------------------------------------------------
    # PAT ENVIRONMENT VARIABLES
    # --------------------------------------------------------
    "pat_name_env": "TABLEAU_PAT_NAME",
    "pat_secret_env": "TABLEAU_PAT_SECRET",

    # --------------------------------------------------------
    # CLOUD EXTRACTION
    # --------------------------------------------------------
    "query_metadata_api": True,
    "download_workbook": True,
    "include_extract": False,

    # --------------------------------------------------------
    # LOCAL WORKBOOK / TWB EXTRACTION
    # --------------------------------------------------------
    "download_dir": "tableau_metadata_work",

    "parse_twb": True,
    "parse_datasources": True,
    "parse_worksheets": True,
    "parse_dashboards": True,
    "parse_calculations": True,
    "parse_parameters": True,
    "parse_filters": True,
    "parse_metadata_records": True,

    "keep_raw_xml": False,

    # --------------------------------------------------------
    # OUTPUT
    # --------------------------------------------------------
    "output_dir": "tableau_metadata_output",

    "export_csv": True,
    "export_json": True,
    "export_excel": True,

    "excel_filename": "tableau_workbook_metadata.xlsx",
    "json_filename": "tableau_workbook_metadata.json",

    "write_manifest": True,

    # --------------------------------------------------------
    # METADATA VERSIONING / CHANGE HISTORY
    # --------------------------------------------------------
    "track_metadata_changes": True,
    "history_dirname": "history",
    "snapshot_dirname": "snapshots",
    "change_log_filename": "tableau_metadata_change_log.csv",
    "state_filename": "latest_metadata_state.json",
    "add_version_columns_to_outputs": True,

    # --------------------------------------------------------
    # VIEW SCREENSHOTS (CLOUD / SERVER)
    # --------------------------------------------------------
    "capture_view_screenshots": True,
    "screenshots_dirname": "view_screenshots",
    "screenshot_width": 1600,
    "screenshot_height": 900,
    "screenshot_max_age_minutes": 1,
}




## Step 2. GLOBAL PATHS / RUN STATE

Initialize working paths, output directories, and runtime state.

In [ ]:
# ============================================================
# 2. GLOBAL PATHS / RUN STATE
# ============================================================

WORK_DIR = Path(CONFIG["download_dir"])
OUTPUT_DIR = Path(CONFIG["output_dir"])

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_STARTED_DT = datetime.now(timezone.utc)
RUN_STARTED_UTC = RUN_STARTED_DT.isoformat()
RUN_ID = RUN_STARTED_DT.strftime("%Y%m%dT%H%M%S") + "Z"
METADATA_VERSION = None

MANIFEST = {
    "run_id": RUN_ID,
    "metadata_version": None,
    "change_log_path": None,
    "snapshot_path": None,
    "screenshots": {"status": "NOT_RUN", "count": 0, "files": [], "errors": []},
    "run_started_utc": RUN_STARTED_UTC,
    "run_finished_utc": None,
    "status": "STARTED",
    "mode": CONFIG["mode"],
    "server": CONFIG["tableau_server"],
    "site_content_url": CONFIG["site_content_url"],
    "api_version": CONFIG["api_version"],
    "workbook_name_requested": CONFIG["workbook_name"],
    "workbook_luid_requested": CONFIG["workbook_luid"],
    "project_path_requested": CONFIG["project_path"],
    "selected_workbook": None,
    "selected_project": None,
    "downloaded_workbook": None,
    "output_files": [],
    "errors": [],
}




## Step 3. VALIDATION

Validate configuration and required settings before extraction.

In [ ]:
# ============================================================
# 3. VALIDATION
# ============================================================

def validate_config():
    mode = str(CONFIG["mode"]).strip().lower()

    if mode not in {"cloud", "local"}:
        raise ValueError(
            f"CONFIG['mode'] must be 'cloud' or 'local'. Got: {CONFIG['mode']!r}"
        )

    if mode == "cloud":
        if not CONFIG["tableau_server"]:
            raise ValueError("CONFIG['tableau_server'] is required.")

        if not CONFIG["site_content_url"]:
            raise ValueError("CONFIG['site_content_url'] is required.")

        if not CONFIG["api_version"]:
            raise ValueError("CONFIG['api_version'] is required.")

        if not CONFIG["workbook_name"] and not CONFIG["workbook_luid"]:
            raise ValueError(
                "For cloud mode, provide either workbook_name or workbook_luid."
            )

        if not CONFIG["project_path"]:
            raise ValueError(
                "CONFIG['project_path'] must contain the expected Tableau "
                "project hierarchy."
            )

    if mode == "local":
        if not CONFIG["local_workbook"]:
            raise ValueError(
                "CONFIG['local_workbook'] is required when mode='local'."
            )

        local_path = Path(CONFIG["local_workbook"])

        if not local_path.exists():
            raise FileNotFoundError(
                f"Local workbook does not exist:\n{local_path}"
            )

        if local_path.suffix.lower() not in {".twb", ".twbx"}:
            raise ValueError(
                "local_workbook must be a .twb or .twbx file."
            )


validate_config()




## Step 4. GENERIC HELPERS

Define shared formatting, parsing, file, and data-processing utilities.

In [ ]:
# ============================================================
# 4. GENERIC HELPERS
# ============================================================

def utc_now():
    return datetime.now(timezone.utc).isoformat()


def normalize_name(value: Any) -> str:
    """
    Normalize names for diagnostic / hierarchy matching.

    Does NOT replace exact matching when Tableau itself can perform
    an exact filter.
    """
    if value is None:
        return ""

    value = str(value)

    value = unicodedata.normalize("NFKC", value)
    value = re.sub(r"\s+", " ", value.strip())

    return value.casefold()


def safe_filename(value: str, max_length: int = 180) -> str:
    value = str(value)

    value = re.sub(r'[<>:"/\\|?*\x00-\x1F]', "_", value)
    value = value.strip().rstrip(".")

    if not value:
        value = "output"

    return value[:max_length]


def safe_excel_sheet_name(
    value: str,
    used_names: Optional[set] = None,
) -> str:
    """
    Excel sheet names:
      - max 31 chars
      - cannot contain []:*?/\
    """
    if used_names is None:
        used_names = set()

    value = str(value)
    value = re.sub(r"[\[\]:*?/\\]", "_", value)
    value = value.strip()

    if not value:
        value = "Sheet"

    value = value[:31]

    base = value
    counter = 1

    while value in used_names:
        suffix = f"_{counter}"
        value = base[:31 - len(suffix)] + suffix
        counter += 1

    used_names.add(value)

    return value


def dataframe_from_records(records: Any) -> pd.DataFrame:
    if records is None:
        return pd.DataFrame()

    if isinstance(records, pd.DataFrame):
        return records.copy()

    if isinstance(records, list):
        if not records:
            return pd.DataFrame()

        return pd.json_normalize(records)

    if isinstance(records, dict):
        return pd.json_normalize([records])

    return pd.DataFrame()


def xml_string(element) -> Optional[str]:
    if not CONFIG["keep_raw_xml"]:
        return None

    try:
        return etree.tostring(
            element,
            encoding="unicode",
            pretty_print=False,
        )
    except Exception:
        return None


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


def strip_namespace(tag: str) -> str:
    if not tag:
        return ""

    if "}" in tag:
        return tag.split("}", 1)[1]

    return tag


def element_attr(element, *names, default=None):
    for name in names:
        value = element.get(name)

        if value is not None:
            return value

    return default


def local_name(element) -> str:
    return strip_namespace(element.tag)




## Step 5. TABLEAU REST CLIENT

Define the Tableau Cloud/Server REST client, authentication, and API operations.

In [ ]:
# ============================================================
# 5. TABLEAU REST CLIENT
# ============================================================

class TableauRestClient:

    def __init__(
        self,
        server: str,
        site_content_url: str,
        api_version: str,
        pat_name: str,
        pat_secret: str,
        timeout: int = 120,
    ):
        self.server = server.rstrip("/")
        self.site_content_url = site_content_url
        self.api_version = api_version

        self.pat_name = pat_name
        self.pat_secret = pat_secret

        self.timeout = timeout

        self.session = requests.Session()

        self.site_id = None
        self.user_id = None
        self.auth_token = None

    # --------------------------------------------------------
    # URL HELPERS
    # --------------------------------------------------------

    @property
    def rest_base(self):
        return f"{self.server}/api/{self.api_version}"

    @property
    def metadata_url(self):
        # IMPORTANT:
        # Tableau Metadata API endpoint is /api/metadata/graphql
        return f"{self.server}/api/metadata/graphql"

    # --------------------------------------------------------
    # REST REQUEST
    # --------------------------------------------------------

    def _headers(self):
        if not self.auth_token:
            raise RuntimeError("Tableau client is not authenticated.")

        return {
            "X-Tableau-Auth": self.auth_token,
            "Accept": "application/json",
        }

    def get(
        self,
        path: str,
        params: Optional[Dict[str, Any]] = None,
    ) -> Dict[str, Any]:
        url = f"{self.server}/api/{self.api_version}{path}"

        response = self.session.get(
            url,
            headers=self._headers(),
            params=params,
            timeout=self.timeout,
        )

        if response.status_code >= 400:
            raise RuntimeError(
                "Tableau REST API GET failed.\n"
                f"HTTP Status: {response.status_code}\n"
                f"URL: {response.url}\n"
                f"Response: {response.text[:2000]}"
            )

        if not response.text.strip():
            return {}

        return response.json()
    # --------------------------------------------------------
    # SIGN IN
    # --------------------------------------------------------

    def signin(self):

        url = f"{self.rest_base}/auth/signin"

        payload = {
            "credentials": {
                "personalAccessTokenName": self.pat_name,
                "personalAccessTokenSecret": self.pat_secret,
                "site": {
                    "contentUrl": self.site_content_url
                },
            }
        }

        response = self.session.post(
            url,
            json=payload,
            headers={
                "Accept": "application/json",
                "Content-Type": "application/json",
            },
            timeout=self.timeout,
        )

        if response.status_code >= 400:
            raise RuntimeError(
                "Tableau sign-in failed.\n\n"
                f"HTTP Status: {response.status_code}\n"
                f"URL: {response.url}\n"
                f"Response:\n{response.text[:5000]}"
            )

        data = response.json()

        credentials = data.get("credentials", {})

        self.auth_token = credentials.get("token")

        site = credentials.get("site", {})

        self.site_id = site.get("id")
        self.site_content_url = site.get(
            "contentUrl",
            self.site_content_url,
        )

        self.user_id = credentials.get("user", {}).get("id")

        if not self.auth_token:
            raise RuntimeError(
                "Tableau sign-in response did not contain an authentication token."
            )

        if not self.site_id:
            raise RuntimeError(
                "Tableau sign-in response did not contain a site ID."
            )

        print("Tableau sign-in successful.")
        print(f"  Site ID: {self.site_id}")
        print(f"  User ID: {self.user_id}")

    # --------------------------------------------------------
    # SIGN OUT
    # --------------------------------------------------------

    def signout(self):

        if not self.auth_token:
            return

        url = f"{self.rest_base}/auth/signout"

        try:
            response = self.session.post(
                url,
                headers=self._headers(),
                timeout=self.timeout,
            )

            if response.status_code not in {200, 204}:
                raise RuntimeError(
                    f"HTTP {response.status_code}: {response.text[:1000]}"
                )

            print("Tableau sign-out successful.")

        finally:
            self.auth_token = None
            self.site_id = None
            self.user_id = None

    # --------------------------------------------------------
    # RESPONSE LIST NORMALIZER
    # --------------------------------------------------------

    @staticmethod
    def _extract_list(
        container: Any,
        singular_key: Optional[str] = None,
    ) -> List[Dict[str, Any]]:

        if container is None:
            return []

        if isinstance(container, list):
            return container

        if isinstance(container, dict):

            if singular_key:
                value = container.get(singular_key)

                if value is None:
                    return []

                if isinstance(value, list):
                    return value

                if isinstance(value, dict):
                    return [value]

                return []

            # Generic dictionary containing one list-like value.
            for value in container.values():

                if isinstance(value, list):
                    return value

                if isinstance(value, dict):
                    return [value]

            return []

        return []

    # --------------------------------------------------------
    # PAGINATION
    # --------------------------------------------------------

    def _get_all_paginated(
        self,
        endpoint: str,
        container_key: str,
        singular_key: str,
        page_size: int = 1000,
        extra_params: Optional[Dict[str, Any]] = None,
    ) -> List[Dict[str, Any]]:

        if page_size < 1 or page_size > 1000:
            raise ValueError("page_size must be between 1 and 1000.")

        extra_params = extra_params or {}

        all_items = []

        page_number = 1
        total_available = None

        while True:

            params = dict(extra_params)

            params["pageSize"] = page_size
            params["pageNumber"] = page_number

            data = self.get(
                endpoint,
                params=params,
            )

            container = data.get(container_key, {})

            items = self._extract_list(
                container,
                singular_key=singular_key,
            )

            all_items.extend(items)

            pagination = data.get("pagination", {})

            if total_available is None:
                try:
                    total_available = int(
                        pagination.get("totalAvailable")
                    )
                except Exception:
                    total_available = None

            if total_available is not None:

                if len(all_items) >= total_available:
                    break

            if len(items) < page_size:
                break

            page_number += 1

        return all_items

    # --------------------------------------------------------
    # PROJECTS
    # --------------------------------------------------------

    def get_projects(self) -> List[Dict[str, Any]]:

        endpoint = f"/sites/{self.site_id}/projects"

        return self._get_all_paginated(
            endpoint=endpoint,
            container_key="projects",
            singular_key="project",
            page_size=1000,
        )

    # --------------------------------------------------------
    # PROJECT HIERARCHY
    # --------------------------------------------------------

    def _project_parent_id(self, project: Dict[str, Any]):
        parent_id = project.get("parentProjectId")

        if parent_id in {"", None}:
            return None

        return parent_id

    def _build_project_path(
        self,
        project: Dict[str, Any],
        projects_by_id: Dict[str, Dict[str, Any]],
    ) -> List[str]:

        names = []

        current = project

        visited = set()

        while current:

            project_id = current.get("id")

            if project_id in visited:
                raise RuntimeError(
                    "Cycle detected while reconstructing Tableau project hierarchy."
                )

            visited.add(project_id)

            names.append(current.get("name", ""))

            parent_id = self._project_parent_id(current)

            if not parent_id:
                break

            current = projects_by_id.get(parent_id)

            if current is None:
                raise RuntimeError(
                    "Unable to resolve parent project while reconstructing "
                    f"hierarchy.\n"
                    f"Project: {project.get('name')}\n"
                    f"Project ID: {project_id}\n"
                    f"Missing parentProjectId: {parent_id}"
                )

        names.reverse()

        return names

    def resolve_project_path(
        self,
        project_path: List[str],
    ) -> Dict[str, Any]:

        if not project_path:
            raise ValueError("project_path cannot be empty.")

        print()
        print("Resolving Tableau project hierarchy...")
        print(
            "  Requested: "
            + " / ".join(project_path)
        )

        projects = self.get_projects()

        if not projects:
            raise RuntimeError(
                "No accessible Tableau projects were returned by the REST API."
            )

        projects_by_id = {
            p.get("id"): p
            for p in projects
            if p.get("id")
        }

        # ----------------------------------------------------
        # Walk hierarchy one level at a time.
        # ----------------------------------------------------

        parent_id = None
        resolved_nodes = []

        for level_index, requested_name in enumerate(project_path):

            normalized_requested = normalize_name(requested_name)

            candidates = []

            for project in projects:

                if normalize_name(project.get("name")) != normalized_requested:
                    continue

                candidate_parent = self._project_parent_id(project)

                if candidate_parent == parent_id:
                    candidates.append(project)

            if not candidates:

                available = []

                for project in projects:

                    candidate_parent = self._project_parent_id(project)

                    if candidate_parent == parent_id:
                        available.append(
                            {
                                "id": project.get("id"),
                                "name": project.get("name"),
                            }
                        )

                available = sorted(
                    available,
                    key=lambda x: normalize_name(x["name"]),
                )

                raise RuntimeError(
                    "Could not resolve Tableau project hierarchy.\n\n"
                    f"Level: {level_index + 1}\n"
                    f"Requested project: {requested_name!r}\n"
                    f"Expected parent ID: {parent_id!r}\n\n"
                    "Available projects at this hierarchy level:\n"
                    + "\n".join(
                        f"  - {x['name']} | ID={x['id']}"
                        for x in available[:100]
                    )
                )

            if len(candidates) > 1:

                details = "\n".join(
                    f"  - {p.get('name')} | "
                    f"ID={p.get('id')} | "
                    f"parentProjectId={p.get('parentProjectId')}"
                    for p in candidates
                )

                raise RuntimeError(
                    "Ambiguous Tableau project hierarchy.\n\n"
                    f"Multiple projects named {requested_name!r} "
                    f"exist under parent ID {parent_id!r}:\n\n"
                    f"{details}\n\n"
                    "The pipeline will not guess which project is correct."
                )

            selected = candidates[0]

            resolved_nodes.append(selected)

            parent_id = selected.get("id")

        final_project = resolved_nodes[-1]

        reconstructed_path = self._build_project_path(
            final_project,
            projects_by_id,
        )

        expected_normalized = [
            normalize_name(x)
            for x in project_path
        ]

        actual_normalized = [
            normalize_name(x)
            for x in reconstructed_path
        ]

        if expected_normalized != actual_normalized:
            raise RuntimeError(
                "Resolved Tableau project hierarchy does not match the "
                "requested hierarchy.\n\n"
                f"Requested: {' / '.join(project_path)}\n"
                f"Resolved:  {' / '.join(reconstructed_path)}"
            )

        print(
            "  Resolved: "
            + " / ".join(reconstructed_path)
        )

        print(
            f"  Final Project ID: {final_project.get('id')}"
        )

        return {
            "project": final_project,
            "nodes": resolved_nodes,
            "path": reconstructed_path,
            "all_projects": projects,
        }

    # --------------------------------------------------------
    # EXACT WORKBOOK NAME
    # --------------------------------------------------------

    def get_workbooks_by_exact_name(
        self,
        workbook_name: str,
    ) -> List[Dict[str, Any]]:

        print()
        print("Locating workbook by exact name...")
        print(f"  Workbook: {workbook_name}")

        endpoint = f"/sites/{self.site_id}/workbooks"

        # ----------------------------------------------------
        # FIRST ATTEMPT:
        # Tableau server-side exact-name filter.
        # ----------------------------------------------------

        filtered = self._get_all_paginated(
            endpoint=endpoint,
            container_key="workbooks",
            singular_key="workbook",
            page_size=1000,
            extra_params={
                "filter": f"name:eq:{workbook_name}",
            },
        )

        # Defensive exact comparison.
        exact = [
            wb
            for wb in filtered
            if wb.get("name") == workbook_name
        ]

        if exact:
            return exact

        # ----------------------------------------------------
        # FALLBACK:
        # Full paginated scan.
        #
        # This protects us if a server-side filter behaves
        # unexpectedly or if the workbook name has a subtle
        # representation issue.
        # ----------------------------------------------------

        print(
            "  Exact-name filter returned no result; "
            "performing full paginated workbook scan..."
        )

        all_workbooks = self._get_all_paginated(
            endpoint=endpoint,
            container_key="workbooks",
            singular_key="workbook",
            page_size=1000,
        )

        exact = [
            wb
            for wb in all_workbooks
            if wb.get("name") == workbook_name
        ]

        if exact:
            return exact

        # ----------------------------------------------------
        # DIAGNOSTICS:
        # normalized near matches.
        # ----------------------------------------------------

        requested_normalized = normalize_name(workbook_name)

        near_matches = []

        for wb in all_workbooks:

            actual_name = wb.get("name", "")

            if normalize_name(actual_name) == requested_normalized:

                near_matches.append(wb)

        if near_matches:

            print()
            print(
                "  WARNING: No exact string match was found, "
                "but normalized-name matches exist:"
            )

            for wb in near_matches[:50]:
                project = wb.get("project", {})

                print(
                    f"    - {wb.get('name')!r} | "
                    f"Workbook ID={wb.get('id')} | "
                    f"Project={project.get('name')} | "
                    f"Project ID={project.get('id')}"
                )

        return []

    # --------------------------------------------------------
    # WORKBOOK SELECTION
    # --------------------------------------------------------

    def get_workbook(
        self,
        workbook_name: Optional[str] = None,
        workbook_luid: Optional[str] = None,
        project_path: Optional[List[str]] = None,
    ) -> Dict[str, Any]:

        workbook_name = (
            workbook_name
            or CONFIG.get("workbook_name")
        )

        workbook_luid = (
            workbook_luid
            or CONFIG.get("workbook_luid")
            or ""
        ).strip()

        project_path = (
            project_path
            if project_path is not None
            else CONFIG.get("project_path")
        )

        resolved_project = None

        if project_path:

            resolved_project = self.resolve_project_path(
                project_path
            )

            target_project = resolved_project["project"]

        else:
            target_project = None

        # ----------------------------------------------------
        # DIRECT LUID LOOKUP
        # ----------------------------------------------------

        if workbook_luid:

            print()
            print("Locating workbook by supplied LUID...")
            print(f"  Workbook LUID: {workbook_luid}")

            endpoint = (
                f"/sites/{self.site_id}/workbooks/{workbook_luid}"
            )

            data = self.get(endpoint)

            workbook = data.get("workbook")

            if not workbook:
                raise RuntimeError(
                    f"Tableau returned no workbook for LUID: "
                    f"{workbook_luid}"
                )

            if target_project:

                actual_project = workbook.get("project", {})

                actual_project_id = actual_project.get("id")
                expected_project_id = target_project.get("id")

                if actual_project_id != expected_project_id:

                    actual_path = (
                        actual_project.get("name")
                        or "<unknown>"
                    )

                    raise RuntimeError(
                        "Supplied workbook LUID does not belong to "
                        "the configured project hierarchy.\n\n"
                        f"Workbook LUID: {workbook_luid}\n"
                        f"Workbook: {workbook.get('name')}\n"
                        f"Actual project: {actual_path}\n"
                        f"Actual project ID: {actual_project_id}\n"
                        f"Expected project path: "
                        f"{' / '.join(project_path)}\n"
                        f"Expected project ID: {expected_project_id}"
                    )

            print("  Workbook found by LUID.")

            return workbook

        # ----------------------------------------------------
        # NAME + PROJECT HIERARCHY
        # ----------------------------------------------------

        if not workbook_name:
            raise RuntimeError(
                "No workbook_name or workbook_luid was supplied."
            )

        candidates = self.get_workbooks_by_exact_name(
            workbook_name
        )

        if not candidates:

            raise RuntimeError(
                "No workbook with the exact name was found.\n\n"
                f"Workbook name: {workbook_name}\n"
                f"Expected project hierarchy: "
                f"{' / '.join(project_path or [])}\n\n"
                "The pipeline intentionally does not guess from "
                "near-name matches."
            )

        # ----------------------------------------------------
        # PROJECT FILTER
        # ----------------------------------------------------

        if target_project:

            expected_project_id = target_project.get("id")

            project_candidates = []

            for wb in candidates:

                wb_project = wb.get("project", {})

                if wb_project.get("id") == expected_project_id:
                    project_candidates.append(wb)

            candidates = project_candidates

        # ----------------------------------------------------
        # ZERO MATCHES AFTER PROJECT FILTER
        # ----------------------------------------------------

        if not candidates:

            raise RuntimeError(
                "Workbook name was found, but no matching workbook "
                "exists in the requested project hierarchy.\n\n"
                f"Workbook name: {workbook_name}\n"
                f"Expected project hierarchy: "
                f"{' / '.join(project_path or [])}\n\n"
                "Exact-name workbook candidates returned by Tableau:"
            )

        # ----------------------------------------------------
        # MULTIPLE MATCHES
        # ----------------------------------------------------

        if len(candidates) > 1:

            details = []

            for wb in candidates:

                project = wb.get("project", {})

                details.append(
                    f"  - Workbook: {wb.get('name')}\n"
                    f"    LUID: {wb.get('id')}\n"
                    f"    Project: {project.get('name')}\n"
                    f"    Project ID: {project.get('id')}\n"
                    f"    Owner: {wb.get('owner', {}).get('name')}"
                )

            raise RuntimeError(
                "Multiple workbooks match the exact workbook name "
                "and project hierarchy.\n\n"
                + "\n".join(details)
                + "\n\n"
                "Set CONFIG['workbook_luid'] explicitly so the "
                "pipeline does not guess."
            )

        workbook = candidates[0]

        print()
        print("Workbook selected successfully.")
        print(f"  Name: {workbook.get('name')}")
        print(f"  LUID: {workbook.get('id')}")

        project = workbook.get("project", {})

        print(f"  Project: {project.get('name')}")
        print(f"  Project ID: {project.get('id')}")

        owner = workbook.get("owner", {})

        print(f"  Owner: {owner.get('name')}")

        return workbook

    # --------------------------------------------------------
    # DOWNLOAD WORKBOOK
    # --------------------------------------------------------

    def download_cloud_workbook(
        self,
        workbook: Dict[str, Any],
    ) -> Path:

        workbook_id = workbook.get("id")

        if not workbook_id:
            raise RuntimeError(
                "Selected workbook does not contain a REST workbook ID/LUID."
            )

        work_dir = WORK_DIR
        work_dir.mkdir(parents=True, exist_ok=True)

        safe_name = safe_filename(
            workbook.get("name", "tableau_workbook")
        )

        print()
        print("Downloading Tableau workbook...")
        print(f"  Destination directory: {work_dir}")
        print(f"  Include extract: {CONFIG['include_extract']}")

        # Tableau can return either a TWB (XML) or TWBX (ZIP).
        # Try compatible Accept headers if a gateway rejects content negotiation.
        base_url = (
            f"{self.rest_base}/sites/{self.site_id}/"
            f"workbooks/{workbook_id}/content"
        )
        auth_header = {"X-Tableau-Auth": self.auth_token}
        include_extract = "true" if CONFIG["include_extract"] else "false"

        attempts = [
            ({"Accept": "*/*"}, {"includeExtract": include_extract}),
            ({"Accept": "application/octet-stream"}, {"includeExtract": include_extract}),
            ({"Accept": "application/xml"}, {"includeExtract": include_extract}),
            ({"Accept": "*/*"}, None),
        ]

        response = None
        last_error = None

        for accept_headers, params in attempts:
            headers = {**auth_header, **accept_headers}
            try:
                candidate = self.session.get(
                    base_url,
                    headers=headers,
                    params=params,
                    timeout=self.timeout,
                    allow_redirects=True,
                )
            except requests.RequestException as exc:
                last_error = f"{type(exc).__name__}: {exc}"
                continue

            if candidate.status_code < 400 and candidate.content:
                response = candidate
                break

            response_summary = (
                f"HTTP {candidate.status_code}; "
                f"Content-Type={candidate.headers.get('Content-Type')!r}; "
                f"URL={candidate.url}; "
                f"Body={candidate.text[:1000]!r}"
            )
            last_error = response_summary

            # Retry only for 406. Other statuses should be surfaced directly.
            if candidate.status_code != 406:
                raise RuntimeError(
                    "Tableau workbook download failed.\n" + response_summary
                )

        if response is None:
            raise RuntimeError(
                "Tableau workbook download failed after content-negotiation retries.\n"
                f"Last response/error: {last_error or 'No response received.'}"
            )

        content = response.content
        if not content:
            raise RuntimeError("Tableau workbook download returned an empty response.")

        # Detect actual format; includeExtract=False can yield a .twb rather than .twbx.
        trimmed = content.lstrip(b"\xef\xbb\xbf\x00\t\r\n ")
        if content.startswith(b"PK\x03\x04"):
            extension = ".twbx"
        elif (
            trimmed.startswith(b"<?xml")
            or trimmed.startswith(b"<workbook")
            or b"<workbook" in trimmed[:1000]
        ):
            extension = ".twb"
        else:
            content_type = response.headers.get("Content-Type", "")
            raise RuntimeError(
                "Tableau returned a successful response, but it does not look like "
                "a TWB XML document or TWBX ZIP archive.\n"
                f"Content-Type: {content_type!r}\n"
                f"First 200 bytes: {content[:200]!r}"
            )

        output_path = work_dir / f"{safe_name}{extension}"
        output_path.write_bytes(content)

        print(f"  Downloaded: {output_path.stat().st_size:,} bytes")
        print(f"  Detected format: {extension[1:].upper()}")
        print(f"  Final URL: {response.url}")

        return output_path

    # --------------------------------------------------------
    # WORKBOOK VIEWS / SCREENSHOT EXPORT
    # --------------------------------------------------------

    def get_workbook_views(self, workbook_id: str) -> List[Dict[str, Any]]:
        """Return all views belonging to a workbook using REST pagination."""
        endpoint = f"/sites/{self.site_id}/workbooks/{workbook_id}/views"
        return self._get_all_paginated(
            endpoint=endpoint,
            container_key="views",
            singular_key="view",
            page_size=1000,
        )

    def download_view_image(self, view_id: str, output_path: Path) -> Path:
        """Download a rendered PNG for a Tableau view."""
        url = (
            f"{self.rest_base}/sites/{self.site_id}/views/"
            f"{view_id}/image"
        )
        params = {
            "maxAge": CONFIG.get("screenshot_max_age_minutes", 1),
            "vizWidth": CONFIG.get("screenshot_width", 1600),
            "vizHeight": CONFIG.get("screenshot_height", 900),
        }
        response = self.session.get(
            url,
            headers={
                "X-Tableau-Auth": self.auth_token,
                "Accept": "image/png,application/octet-stream,*/*",
            },
            params=params,
            timeout=self.timeout,
        )
        if response.status_code >= 400:
            raise RuntimeError(
                f"View image export failed for view {view_id}. "
                f"HTTP {response.status_code}: {response.text[:1000]}"
            )
        if not response.content:
            raise RuntimeError(f"View image export returned empty content for {view_id}.")
        content_type = response.headers.get("Content-Type", "").lower()
        if "json" in content_type or "xml" in content_type:
            raise RuntimeError(
                f"View image endpoint returned {content_type}, not an image."
            )
        output_path.parent.mkdir(parents=True, exist_ok=True)
        output_path.write_bytes(response.content)
        return output_path

    # --------------------------------------------------------
    # METADATA API
    # --------------------------------------------------------

    def metadata_query(
        self,
        query: str,
        variables: Optional[Dict[str, Any]] = None,
    ) -> Dict[str, Any]:

        if not self.auth_token:
            raise RuntimeError(
                "Cannot query Metadata API before Tableau sign-in."
            )

        # ----------------------------------------------------
        # CRITICAL:
        # Correct Tableau Metadata API endpoint.
        #
        # NOT:
        #   /graphql
        #
        # NOT:
        #   /metadata/graphql
        #
        # YES:
        #   /api/metadata/graphql
        # ----------------------------------------------------

        url = self.metadata_url

        payload = {
            "query": query,
        }

        if variables is not None:
            payload["variables"] = variables

        response = self.session.post(
            url,
            headers={
                "X-Tableau-Auth": self.auth_token,
                "Accept": "application/json",
                "Content-Type": "application/json",
            },
            json=payload,
            timeout=self.timeout,
        )

        if response.status_code >= 400:
            raise RuntimeError(
                "Tableau Metadata API request failed.\n\n"
                f"HTTP Status: {response.status_code}\n"
                f"URL: {response.url}\n"
                f"Response:\n{response.text[:10000]}"
            )

        try:
            result = response.json()

        except Exception as exc:
            raise RuntimeError(
                "Tableau Metadata API returned a non-JSON response.\n\n"
                f"HTTP Status: {response.status_code}\n"
                f"URL: {response.url}\n"
                f"Response:\n{response.text[:10000]}"
            ) from exc

        if result.get("errors"):

            raise RuntimeError(
                "Tableau Metadata API returned GraphQL errors.\n\n"
                f"{json.dumps(result['errors'], indent=2, ensure_ascii=False)}"
            )

        return result.get("data", result)

    # --------------------------------------------------------
    # WORKBOOK METADATA QUERY
    # --------------------------------------------------------

    def query_workbook_metadata(
        self,
        workbook_luid: str,
    ) -> Dict[str, Any]:

        if not workbook_luid:
            raise ValueError(
                "workbook_luid is required for Metadata API extraction."
            )

        print()
        print("Querying Tableau Metadata API...")
        print(f"  Workbook LUID: {workbook_luid}")
        print(f"  Endpoint: {self.metadata_url}")

        # ----------------------------------------------------
        # The Metadata API supports filtering Workbook by LUID.
        #
        # This is deliberately NOT filtered by workbook name,
        # because duplicate workbook names exist on the site.
        # ----------------------------------------------------

        query = """
        query WorkbookMetadata($workbookLuid: String!) {
          workbooks(filter: {luid: $workbookLuid}) {
            id
            luid
            name
            containsUnsupportedCustomSql
            projectVizportalUrlId
            projectName
            projectLuid
            containerType
            containerName

            owner {
              id
              luid
              username
              name
              domain
              email
            }

            tags {
              id
              name
            }

            sheets {
              id
              luid
              name
              documentViewId
              path
              createdAt
              updatedAt
              index
              tags {
                id
                name
              }

              worksheetFields {
                id
                name
                fullyQualifiedName
                description
                aggregation
                aggregationParam
                formula
                isAutoGenerated
                hasUserReference
              }

              datasourceFields {
                id
                name
              }
            }

            dashboards {
              id
              luid
              name
              documentViewId
              path
              createdAt
              updatedAt
              index
              tags {
                id
                name
              }
            }

            embeddedDatasources {
              id
              name
              hasUserReference
              hasExtracts
              containsUnsupportedCustomSql

              fields {
                id
                name
              }
            }

            parameters {
              id
              name
            }
          }
        }
        """

        data = self.metadata_query(
            query=query,
            variables={
                "workbookLuid": workbook_luid,
            },
        )

        workbooks = data.get("workbooks", [])

        if not workbooks:

            raise RuntimeError(
                "Metadata API returned no workbook for the selected "
                f"REST workbook LUID: {workbook_luid}\n\n"
                "Possible causes:\n"
                "  1. Metadata API / Tableau Catalog is not enabled.\n"
                "  2. The authenticated PAT/user lacks required access.\n"
                "  3. Tableau metadata indexing has not caught up yet.\n"
                "  4. The workbook is not visible to the authenticated user."
            )

        if len(workbooks) > 1:

            raise RuntimeError(
                "Metadata API returned more than one workbook for "
                f"LUID {workbook_luid}, which should not happen."
            )

        workbook_metadata = workbooks[0]

        print("Metadata API query successful.")

        return workbook_metadata




## Step 6. LOCAL TWB / TWBX LOADER

Load workbook files and parse their Tableau XML content.

In [ ]:
# ============================================================
# 6. LOCAL TWB / TWBX LOADER
# ============================================================

def load_twb_from_path(
    path: Path,
) -> Tuple[Path, bytes, etree._ElementTree, Dict[str, Any]]:

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(path)

    suffix = path.suffix.lower()

    if suffix == ".twb":

        xml_bytes = path.read_bytes()

        xml_root = etree.fromstring(xml_bytes)

        tree = etree.ElementTree(xml_root)

        info = {
            "source_path": str(path),
            "source_type": "twb",
            "sha256": sha256_file(path),
            "twb_member": path.name,
        }

        return path, xml_bytes, tree, info

    if suffix == ".twbx":

        with zipfile.ZipFile(path, "r") as z:

            members = z.namelist()

            twb_members = [
                m
                for m in members
                if m.lower().endswith(".twb")
            ]

            if not twb_members:
                raise RuntimeError(
                    f"No .twb file was found inside:\n{path}"
                )

            # Prefer a root-level TWB if available.
            twb_members.sort(
                key=lambda x: (
                    "/" in x,
                    x.count("/"),
                    x.lower(),
                )
            )

            twb_member = twb_members[0]

            xml_bytes = z.read(twb_member)

        xml_root = etree.fromstring(xml_bytes)

        tree = etree.ElementTree(xml_root)

        info = {
            "source_path": str(path),
            "source_type": "twbx",
            "sha256": sha256_file(path),
            "twb_member": twb_member,
        }

        return path, xml_bytes, tree, info

    raise ValueError(
        f"Unsupported Tableau workbook type: {suffix}"
    )




## Step 7. TWB XML EXTRACTOR

Extract workbook metadata from the Tableau XML structure.

In [ ]:
# ============================================================
# 7. TWB XML EXTRACTOR
# ============================================================

class TWBExtractor:

    def __init__(
        self,
        tree: etree._ElementTree,
        source_info: Dict[str, Any],
    ):

        self.tree = tree
        self.root = tree.getroot()
        self.source_info = source_info

        self.tables = {}

    # --------------------------------------------------------
    # GENERIC XML WALKER
    # --------------------------------------------------------

    def iter_elements(self, tag_name: Optional[str] = None):

        for element in self.root.iter():

            if tag_name is None:
                yield element

            elif local_name(element) == tag_name:
                yield element

    # --------------------------------------------------------
    # WORKBOOK
    # --------------------------------------------------------

    def extract_workbook(self):

        rows = []

        root = self.root

        row = {
            "source_type": self.source_info.get("source_type"),
            "source_path": self.source_info.get("source_path"),
            "twb_member": self.source_info.get("twb_member"),
            "sha256": self.source_info.get("sha256"),
            "workbook_version": element_attr(
                root,
                "version",
            ),
            "xml_root_tag": local_name(root),
        }

        rows.append(row)

        self.tables["workbook"] = rows

    # --------------------------------------------------------
    # DATASOURCES
    # --------------------------------------------------------

    def extract_datasources(self):

        rows = []

        for element in self.iter_elements("datasource"):

            row = {
                "name": element_attr(
                    element,
                    "name",
                ),
                "caption": element_attr(
                    element,
                    "caption",
                ),
                "hasconnection": element_attr(
                    element,
                    "hasconnection",
                ),
                "inline": element_attr(
                    element,
                    "inline",
                ),
                "version": element_attr(
                    element,
                    "version",
                ),
                "object_model_encapsulate": element_attr(
                    element,
                    "object-model-encapsulate",
                ),
                "raw_xml": xml_string(element),
            }

            rows.append(row)

        self.tables["datasources"] = rows

    # --------------------------------------------------------
    # CONNECTIONS
    # --------------------------------------------------------

    def extract_connections(self):

        rows = []

        for datasource in self.iter_elements("datasource"):

            datasource_name = element_attr(
                datasource,
                "name",
            )

            for connection in datasource:

                if local_name(connection) != "connection":
                    continue

                row = {
                    "datasource": datasource_name,
                    "connection_type": element_attr(
                        connection,
                        "class",
                        "type",
                    ),
                    "server": element_attr(
                        connection,
                        "server",
                    ),
                    "database": element_attr(
                        connection,
                        "dbname",
                        "database",
                    ),
                    "schema": element_attr(
                        connection,
                        "schema",
                    ),
                    "authentication": element_attr(
                        connection,
                        "authentication",
                    ),
                    "port": element_attr(
                        connection,
                        "port",
                    ),
                    "username": element_attr(
                        connection,
                        "username",
                    ),
                    "filename": element_attr(
                        connection,
                        "filename",
                    ),
                    "raw_xml": xml_string(connection),
                }

                rows.append(row)

        self.tables["connections"] = rows

    # --------------------------------------------------------
    # FIELDS
    # --------------------------------------------------------

    def extract_fields(self):

        rows = []

        for datasource in self.iter_elements("datasource"):

            datasource_name = element_attr(
                datasource,
                "name",
            )

            for element in datasource.iter():

                if local_name(element) != "column":
                    continue

                row = {
                    "datasource": datasource_name,
                    "name": element_attr(
                        element,
                        "name",
                    ),
                    "caption": element_attr(
                        element,
                        "caption",
                    ),
                    "datatype": element_attr(
                        element,
                        "datatype",
                        "data-type",
                    ),
                    "role": element_attr(
                        element,
                        "role",
                    ),
                    "type": element_attr(
                        element,
                        "type",
                    ),
                    "datatype_source": element_attr(
                        element,
                        "datatype",
                    ),
                    "hidden": element_attr(
                        element,
                        "hidden",
                    ),
                    "aggregation": element_attr(
                        element,
                        "aggregation",
                    ),
                    "default_format": element_attr(
                        element,
                        "default-format",
                    ),
                    "semantic_role": element_attr(
                        element,
                        "semantic-role",
                    ),
                    "folder": element_attr(
                        element,
                        "folder",
                    ),
                    "raw_xml": xml_string(element),
                }

                rows.append(row)

        self.tables["fields"] = rows

    # --------------------------------------------------------
    # CALCULATIONS
    # --------------------------------------------------------

    def extract_calculations(self):

        rows = []

        for element in self.iter_elements("column"):

            calculation = None

            for child in element:

                if local_name(child) == "calculation":
                    calculation = child
                    break

            if calculation is None:
                continue

            row = {
                "name": element_attr(
                    element,
                    "name",
                ),
                "caption": element_attr(
                    element,
                    "caption",
                ),
                "datatype": element_attr(
                    element,
                    "datatype",
                    "data-type",
                ),
                "role": element_attr(
                    element,
                    "role",
                ),
                "calculation_class": element_attr(
                    calculation,
                    "class",
                ),
                "formula": element_attr(
                    calculation,
                    "formula",
                ),
                "raw_xml": xml_string(element),
            }

            rows.append(row)

        self.tables["calculations"] = rows

    # --------------------------------------------------------
    # ALIASES
    # --------------------------------------------------------

    def extract_aliases(self):

        rows = []

        for element in self.iter_elements("aliases"):

            parent = element.getparent()

            row = {
                "field_name": (
                    element_attr(
                        parent,
                        "name",
                    )
                    if parent is not None
                    else None
                ),
                "raw_xml": xml_string(element),
            }

            for alias in element:

                if local_name(alias) != "alias":
                    continue

                row["field_value"] = element_attr(
                    alias,
                    "key",
                    "value",
                )

                row["alias"] = element_attr(
                    alias,
                    "value",
                    "alias",
                )

                rows.append(dict(row))

        self.tables["aliases"] = rows

    # --------------------------------------------------------
    # WORKSHEETS
    # --------------------------------------------------------

    def extract_worksheets(self):

        rows = []

        worksheets = self.root.xpath(
            ".//*[local-name()='worksheet']"
        )

        for worksheet in worksheets:

            row = {
                "name": element_attr(
                    worksheet,
                    "name",
                ),
                "caption": element_attr(
                    worksheet,
                    "caption",
                ),
                "repository_location": element_attr(
                    worksheet,
                    "repository-location",
                ),
                "raw_xml": xml_string(worksheet),
            }

            rows.append(row)

        self.tables["worksheets"] = rows

    # --------------------------------------------------------
    # FILTERS
    # --------------------------------------------------------

    def extract_filters(self):

        rows = []

        for element in self.iter_elements("filter"):

            parent = element.getparent()

            row = {
                "filter_type": element_attr(
                    element,
                    "class",
                    "type",
                ),
                "column": element_attr(
                    element,
                    "column",
                    "field",
                ),
                "member": element_attr(
                    element,
                    "member",
                ),
                "formula": element_attr(
                    element,
                    "formula",
                ),
                "parent_tag": (
                    local_name(parent)
                    if parent is not None
                    else None
                ),
                "parent_name": (
                    element_attr(
                        parent,
                        "name",
                    )
                    if parent is not None
                    else None
                ),
                "raw_xml": xml_string(element),
            }

            rows.append(row)

        self.tables["filters"] = rows

    # --------------------------------------------------------
    # DASHBOARDS
    # --------------------------------------------------------

    def extract_dashboards(self):

        rows = []

        dashboards = self.root.xpath(
            ".//*[local-name()='dashboard']"
        )

        for dashboard in dashboards:

            row = {
                "name": element_attr(
                    dashboard,
                    "name",
                ),
                "caption": element_attr(
                    dashboard,
                    "caption",
                ),
                "showTabs": element_attr(
                    dashboard,
                    "showTabs",
                ),
                "raw_xml": xml_string(dashboard),
            }

            rows.append(row)

        self.tables["dashboards"] = rows

    # --------------------------------------------------------
    # DASHBOARD ZONES
    # --------------------------------------------------------

    def extract_dashboard_zones(self):

        rows = []

        dashboards = self.root.xpath(
            ".//*[local-name()='dashboard']"
        )

        for dashboard in dashboards:

            dashboard_name = element_attr(
                dashboard,
                "name",
            )

            for zone in dashboard.iter():

                if local_name(zone) not in {
                    "zone",
                    "zone-style",
                }:
                    continue

                row = {
                    "dashboard": dashboard_name,
                    "zone_id": element_attr(
                        zone,
                        "id",
                    ),
                    "name": element_attr(
                        zone,
                        "name",
                    ),
                    "type": element_attr(
                        zone,
                        "type",
                        "name",
                    ),
                    "x": element_attr(
                        zone,
                        "x",
                    ),
                    "y": element_attr(
                        zone,
                        "y",
                    ),
                    "w": element_attr(
                        zone,
                        "w",
                    ),
                    "h": element_attr(
                        zone,
                        "h",
                    ),
                    "raw_xml": xml_string(zone),
                }

                rows.append(row)

        self.tables["dashboard_zones"] = rows

    # --------------------------------------------------------
    # PARAMETERS
    # --------------------------------------------------------

    def extract_parameters(self):

        rows = []

        for element in self.iter_elements("column"):

            parameter_domain = None

            for child in element:

                if local_name(child) == "calculation":
                    continue

                if local_name(child) in {
                    "range",
                    "members",
                    "parameter-domain",
                }:
                    parameter_domain = child
                    break

            # Tableau parameter columns commonly have
            # parameter attributes.
            is_parameter = (
                element_attr(
                    element,
                    "param-domain-type",
                    "parameter-domain-type",
                )
                is not None
                or parameter_domain is not None
            )

            if not is_parameter:
                continue

            rows.append(
                {
                    "name": element_attr(
                        element,
                        "name",
                    ),
                    "caption": element_attr(
                        element,
                        "caption",
                    ),
                    "datatype": element_attr(
                        element,
                        "datatype",
                    ),
                    "param_domain_type": element_attr(
                        element,
                        "param-domain-type",
                        "parameter-domain-type",
                    ),
                    "value": element_attr(
                        element,
                        "value",
                    ),
                    "raw_xml": xml_string(element),
                }
            )

        self.tables["parameters"] = rows

    # --------------------------------------------------------
    # CUSTOM SQL
    # --------------------------------------------------------

    def extract_custom_sql(self):

        rows = []

        for element in self.iter_elements("relation"):

            if element_attr(
                element,
                "type",
            ) != "text":
                continue

            text = "".join(
                element.itertext()
            ).strip()

            if not text:
                continue

            rows.append(
                {
                    "datasource": (
                        element.getparent().getparent().get("name")
                        if element.getparent() is not None
                        and element.getparent().getparent() is not None
                        else None
                    ),
                    "text": text,
                    "raw_xml": xml_string(element),
                }
            )

        self.tables["custom_sql"] = rows

    # --------------------------------------------------------
    # GROUPS
    # --------------------------------------------------------

    def extract_groups(self):

        rows = []

        for element in self.iter_elements("group"):

            rows.append(
                {
                    "name": element_attr(
                        element,
                        "name",
                    ),
                    "caption": element_attr(
                        element,
                        "caption",
                    ),
                    "raw_xml": xml_string(element),
                }
            )

        self.tables["groups"] = rows

    # --------------------------------------------------------
    # SETS
    # --------------------------------------------------------

    def extract_sets(self):

        rows = []

        for element in self.iter_elements("set"):

            rows.append(
                {
                    "name": element_attr(
                        element,
                        "name",
                    ),
                    "caption": element_attr(
                        element,
                        "caption",
                    ),
                    "raw_xml": xml_string(element),
                }
            )

        self.tables["sets"] = rows

    # --------------------------------------------------------
    # HIERARCHIES
    # --------------------------------------------------------

    def extract_hierarchies(self):

        rows = []

        for element in self.iter_elements("hierarchy"):

            rows.append(
                {
                    "name": element_attr(
                        element,
                        "name",
                    ),
                    "caption": element_attr(
                        element,
                        "caption",
                    ),
                    "raw_xml": xml_string(element),
                }
            )

        self.tables["hierarchies"] = rows

    # --------------------------------------------------------
    # BINS
    # --------------------------------------------------------

    def extract_bins(self):

        rows = []

        for element in self.iter_elements("bin"):

            rows.append(
                {
                    "name": element_attr(
                        element,
                        "name",
                    ),
                    "source": element_attr(
                        element,
                        "source",
                    ),
                    "raw_xml": xml_string(element),
                }
            )

        self.tables["bins"] = rows

    # --------------------------------------------------------
    # RUN
    # --------------------------------------------------------

    def run(self) -> Dict[str, List[Dict[str, Any]]]:

        if CONFIG["parse_twb"]:
            self.extract_workbook()

        if CONFIG["parse_datasources"]:
            self.extract_datasources()
            self.extract_connections()
            self.extract_fields()
            self.extract_aliases()
            self.extract_custom_sql()
            self.extract_groups()
            self.extract_sets()
            self.extract_hierarchies()
            self.extract_bins()

        if CONFIG["parse_worksheets"]:
            self.extract_worksheets()

        if CONFIG["parse_dashboards"]:
            self.extract_dashboards()
            self.extract_dashboard_zones()

        if CONFIG["parse_calculations"]:
            self.extract_calculations()

        if CONFIG["parse_parameters"]:
            self.extract_parameters()

        if CONFIG["parse_filters"]:
            self.extract_filters()

        return self.tables




## Step 8. METADATA API NORMALIZATION

Normalize Tableau Metadata API responses into tabular records.

In [ ]:
# ============================================================
# 8. METADATA API NORMALIZATION
# ============================================================

def normalize_metadata_api(
    metadata: Dict[str, Any],
) -> Dict[str, List[Dict[str, Any]]]:

    tables = {}

    # --------------------------------------------------------
    # Workbook
    # --------------------------------------------------------

    workbook_record = {
        "id": metadata.get("id"),
        "luid": metadata.get("luid"),
        "name": metadata.get("name"),
        "containsUnsupportedCustomSql": metadata.get(
            "containsUnsupportedCustomSql"
        ),
        "projectVizportalUrlId": metadata.get(
            "projectVizportalUrlId"
        ),
        "projectName": metadata.get(
            "projectName"
        ),
        "projectLuid": metadata.get(
            "projectLuid"
        ),
        "containerType": metadata.get(
            "containerType"
        ),
        "containerName": metadata.get(
            "containerName"
        ),
    }

    owner = metadata.get("owner") or {}

    workbook_record.update(
        {
            "owner_id": owner.get("id"),
            "owner_luid": owner.get("luid"),
            "owner_username": owner.get("username"),
            "owner_name": owner.get("name"),
            "owner_domain": owner.get("domain"),
            "owner_email": owner.get("email"),
        }
    )

    tables["cloud_workbook"] = [workbook_record]

    # --------------------------------------------------------
    # Workbook tags
    # --------------------------------------------------------

    tables["cloud_workbook_tags"] = [
        {
            "workbook_luid": metadata.get("luid"),
            "tag_id": tag.get("id"),
            "tag_name": tag.get("name"),
        }
        for tag in metadata.get("tags", [])
        if isinstance(tag, dict)
    ]

    # --------------------------------------------------------
    # Sheets
    # --------------------------------------------------------

    sheet_rows = []

    for sheet in metadata.get("sheets", []):

        if not isinstance(sheet, dict):
            continue

        base = {
            "workbook_luid": metadata.get("luid"),
            "sheet_id": sheet.get("id"),
            "sheet_luid": sheet.get("luid"),
            "name": sheet.get("name"),
            "documentViewId": sheet.get("documentViewId"),
            "path": sheet.get("path"),
            "createdAt": sheet.get("createdAt"),
            "updatedAt": sheet.get("updatedAt"),
            "index": sheet.get("index"),
        }

        tags = sheet.get("tags") or []

        if tags:
            for tag in tags:
                row = dict(base)
                row["tag_id"] = tag.get("id")
                row["tag_name"] = tag.get("name")
                sheet_rows.append(row)

        else:
            sheet_rows.append(base)

    tables["cloud_sheets"] = sheet_rows

    # --------------------------------------------------------
    # Dashboards
    # --------------------------------------------------------

    dashboard_rows = []

    for dashboard in metadata.get("dashboards", []):

        if not isinstance(dashboard, dict):
            continue

        dashboard_rows.append(
            {
                "workbook_luid": metadata.get("luid"),
                "dashboard_id": dashboard.get("id"),
                "dashboard_luid": dashboard.get("luid"),
                "name": dashboard.get("name"),
                "documentViewId": dashboard.get(
                    "documentViewId"
                ),
                "path": dashboard.get("path"),
                "createdAt": dashboard.get("createdAt"),
                "updatedAt": dashboard.get("updatedAt"),
                "index": dashboard.get("index"),
            }
        )

    tables["cloud_dashboards"] = dashboard_rows

    # --------------------------------------------------------
    # Embedded datasources
    # --------------------------------------------------------

    datasource_rows = []

    for datasource in metadata.get(
        "embeddedDatasources",
        [],
    ):

        if not isinstance(datasource, dict):
            continue

        datasource_rows.append(
            {
                "workbook_luid": metadata.get("luid"),
                "datasource_id": datasource.get("id"),
                "name": datasource.get("name"),
                "hasUserReference": datasource.get(
                    "hasUserReference"
                ),
                "hasExtracts": datasource.get(
                    "hasExtracts"
                ),
                "containsUnsupportedCustomSql": datasource.get(
                    "containsUnsupportedCustomSql"
                ),
            }
        )

    tables["cloud_datasources"] = datasource_rows

    # --------------------------------------------------------
    # Datasource fields
    # --------------------------------------------------------

    field_rows = []

    for datasource in metadata.get(
        "embeddedDatasources",
        [],
    ):

        if not isinstance(datasource, dict):
            continue

        datasource_id = datasource.get("id")
        datasource_name = datasource.get("name")

        for field in datasource.get("fields", []) or []:

            if not isinstance(field, dict):
                continue

            field_rows.append(
                {
                    "workbook_luid": metadata.get("luid"),
                    "datasource_id": datasource_id,
                    "datasource_name": datasource_name,
                    "field_id": field.get("id"),
                    "field_name": field.get("name"),
                    "fullyQualifiedName": field.get(
                        "fullyQualifiedName"
                    ),
                    "description": field.get(
                        "description"
                    ),
                    "role": field.get("role"),
                    "dataType": field.get("dataType"),
                }
            )

    tables["cloud_datasource_fields"] = field_rows

    # --------------------------------------------------------
    # Worksheet calculated fields
    # --------------------------------------------------------

    calculation_rows = []

    for sheet in metadata.get("sheets", []):

        if not isinstance(sheet, dict):
            continue

        sheet_id = sheet.get("id")
        sheet_name = sheet.get("name")

        for field in sheet.get(
            "worksheetFields",
            [],
        ) or []:

            if not isinstance(field, dict):
                continue

            calculation_rows.append(
                {
                    "workbook_luid": metadata.get("luid"),
                    "sheet_id": sheet_id,
                    "sheet_name": sheet_name,
                    "field_id": field.get("id"),
                    "field_name": field.get("name"),
                    "fullyQualifiedName": field.get(
                        "fullyQualifiedName"
                    ),
                    "description": field.get(
                        "description"
                    ),
                    "aggregation": field.get(
                        "aggregation"
                    ),
                    "aggregationParam": field.get(
                        "aggregationParam"
                    ),
                    "formula": field.get("formula"),
                    "isAutoGenerated": field.get(
                        "isAutoGenerated"
                    ),
                    "hasUserReference": field.get(
                        "hasUserReference"
                    ),
                }
            )

    tables["cloud_calculated_fields"] = calculation_rows

    # --------------------------------------------------------
    # Worksheet datasource fields
    # --------------------------------------------------------

    worksheet_field_rows = []

    for sheet in metadata.get("sheets", []):

        if not isinstance(sheet, dict):
            continue

        for field in sheet.get(
            "datasourceFields",
            [],
        ) or []:

            if not isinstance(field, dict):
                continue

            worksheet_field_rows.append(
                {
                    "workbook_luid": metadata.get("luid"),
                    "sheet_id": sheet.get("id"),
                    "sheet_name": sheet.get("name"),
                    "field_id": field.get("id"),
                    "field_name": field.get("name"),
                    "fullyQualifiedName": field.get(
                        "fullyQualifiedName"
                    ),
                    "description": field.get(
                        "description"
                    ),
                    "role": field.get("role"),
                    "dataType": field.get("dataType"),
                }
            )

    tables["cloud_worksheet_datasource_fields"] = (
        worksheet_field_rows
    )

    # --------------------------------------------------------
    # Parameters
    # --------------------------------------------------------

    parameter_rows = []

    for parameter in metadata.get(
        "parameters",
        [],
    ):

        if not isinstance(parameter, dict):
            continue

        parameter_rows.append(
            {
                "workbook_luid": metadata.get("luid"),
                "parameter_id": parameter.get("id"),
                "name": parameter.get("name"),
                "description": parameter.get(
                    "description"
                ),
            }
        )

    tables["cloud_parameters"] = parameter_rows

    return tables




## Step 9. COMBINE / QUALITY ASSURANCE

Combine metadata sources and build quality-assurance summaries.

In [ ]:
# ============================================================
# 9. COMBINE / QUALITY ASSURANCE
# ============================================================

def build_summary(
    tables: Dict[str, pd.DataFrame],
) -> pd.DataFrame:

    rows = []

    for name, df in tables.items():

        rows.append(
            {
                "table": name,
                "row_count": len(df),
                "column_count": len(df.columns),
                "empty": len(df) == 0,
            }
        )

    return pd.DataFrame(rows)


def build_qa(
    tables: Dict[str, pd.DataFrame],
    selected_workbook: Optional[Dict[str, Any]] = None,
    selected_project: Optional[Dict[str, Any]] = None,
) -> pd.DataFrame:

    rows = []

    expected_workbook_name = CONFIG.get(
        "workbook_name"
    )

    selected_luid = None

    if selected_workbook:
        selected_luid = selected_workbook.get("id")

    # --------------------------------------------------------
    # Workbook identity
    # --------------------------------------------------------

    if selected_workbook:

        actual_name = selected_workbook.get("name")

        rows.append(
            {
                "check": "REST workbook selected",
                "status": (
                    "PASS"
                    if actual_name == expected_workbook_name
                    else "WARN"
                ),
                "detail": (
                    f"name={actual_name!r}; "
                    f"LUID={selected_luid}"
                ),
            }
        )

    # --------------------------------------------------------
    # Project identity
    # --------------------------------------------------------

    if selected_project:

        expected_path = " / ".join(
            CONFIG.get("project_path", [])
        )

        rows.append(
            {
                "check": "REST project resolved",
                "status": "PASS",
                "detail": (
                    f"path={expected_path}; "
                    f"project_id={selected_project.get('id')}; "
                    f"project_name={selected_project.get('name')}"
                ),
            }
        )

    # --------------------------------------------------------
    # LUID consistency
    # --------------------------------------------------------

    if selected_luid and "cloud_workbook" in tables:

        df = tables["cloud_workbook"]

        if not df.empty:

            metadata_luid = df.iloc[0].get("luid")

            rows.append(
                {
                    "check": "REST LUID vs Metadata API LUID",
                    "status": (
                        "PASS"
                        if metadata_luid == selected_luid
                        else "FAIL"
                    ),
                    "detail": (
                        f"REST={selected_luid}; "
                        f"Metadata={metadata_luid}"
                    ),
                }
            )

    # --------------------------------------------------------
    # Workbook file download status
    # --------------------------------------------------------

    if CONFIG.get("mode", "").lower() == "cloud" and CONFIG.get("download_workbook"):
        current_download_error = globals().get("download_error")
        rows.append(
            {
                "check": "Cloud workbook file download",
                "status": "WARN" if current_download_error else "PASS",
                "detail": (
                    current_download_error
                    if current_download_error
                    else "Workbook file downloaded and parsed if parsing was enabled."
                ),
            }
        )

    # --------------------------------------------------------
    # Basic table checks
    # --------------------------------------------------------

    for name, df in tables.items():

        rows.append(
            {
                "check": f"Table generated: {name}",
                "status": "PASS",
                "detail": (
                    f"rows={len(df)}, "
                    f"columns={len(df.columns)}"
                ),
            }
        )

    return pd.DataFrame(rows)




## Step 10. METADATA VERSIONING / CHANGE LOG

Maintain snapshots, metadata versions, and record-level change history.

In [ ]:
# ============================================================
# 10. METADATA VERSIONING / CHANGE LOG
# ============================================================

def _canonical_json(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )


def _row_hash(record):
    return hashlib.sha256(_canonical_json(record).encode("utf-8")).hexdigest()


def _record_identity(table_name, record, record_hash):
    """Choose a practical stable key; fall back to content hash if no key exists."""
    lower = {str(k).casefold(): k for k in record.keys()}
    for candidate in ("id", "luid", "field id", "view id", "workbook id", "datasource id"):
        if candidate in lower and record.get(lower[candidate]) not in (None, ""):
            return f"{candidate}:{record[lower[candidate]]}"

    # Composite names are often the best available identity in TWB-derived tables.
    name_key = next((lower[k] for k in ("name", "caption", "field name", "sheet name", "view name") if k in lower), None)
    if name_key is not None and record.get(name_key) not in (None, ""):
        parent_parts = []
        for k in ("workbook name", "datasource name", "data source", "worksheet", "sheet", "dashboard", "parent", "project name"):
            if k in lower and record.get(lower[k]) not in (None, ""):
                parent_parts.append(f"{k}={record[lower[k]]}")
        return f"name:{record[name_key]}|" + "|".join(parent_parts)
    return f"content:{record_hash}"


def track_metadata_history(tables):
    """Snapshot extracted metadata, compare with previous run, append a change log."""
    global METADATA_VERSION

    if not CONFIG.get("track_metadata_changes", True):
        return tables

    history_dir = OUTPUT_DIR / CONFIG.get("history_dirname", "history")
    snapshots_dir = history_dir / CONFIG.get("snapshot_dirname", "snapshots")
    snapshots_dir.mkdir(parents=True, exist_ok=True)
    state_path = history_dir / CONFIG.get("state_filename", "latest_metadata_state.json")
    change_log_path = history_dir / CONFIG.get("change_log_filename", "tableau_metadata_change_log.csv")
    history_dir.mkdir(parents=True, exist_ok=True)

    previous_state = {}
    previous_version = 0
    if state_path.exists():
        try:
            previous_state = json.loads(state_path.read_text(encoding="utf-8"))
            previous_version = int(previous_state.get("metadata_version", 0))
        except Exception as exc:
            raise RuntimeError(
                f"Could not read metadata history state at {state_path}. "
                "Preserve this file and inspect it before rerunning."
            ) from exc

    METADATA_VERSION = previous_version + 1
    extracted_at = utc_now()
    change_rows = []
    current_state = {
        "run_id": RUN_ID,
        "metadata_version": METADATA_VERSION,
        "extracted_at_utc": extracted_at,
        "workbook": MANIFEST.get("selected_workbook") or {
            "name": CONFIG.get("workbook_name"),
            "luid": CONFIG.get("workbook_luid"),
        },
        "tables": {},
    }

    for table_name, df in tables.items():
        records = df.to_dict(orient="records") if isinstance(df, pd.DataFrame) else list(df or [])
        current_records = {}
        for record in records:
            normalized = {}
            for k, v in dict(record).items():
                try:
                    is_missing = pd.isna(v)
                    if isinstance(is_missing, (bool, np.bool_)) and is_missing:
                        v = None
                except Exception:
                    pass
                normalized[str(k)] = v
            digest = _row_hash(normalized)
            identity = _record_identity(table_name, normalized, digest)
            # Avoid silently overwriting duplicate identities.
            if identity in current_records:
                identity = f"{identity}#duplicate-{len(current_records)}"
            current_records[identity] = {
                "hash": digest,
                "record": normalized,
            }

        old_records = previous_state.get("tables", {}).get(table_name, {})
        for identity in sorted(set(old_records) | set(current_records)):
            old = old_records.get(identity)
            new = current_records.get(identity)
            if old is None:
                change_type = "ADDED"
            elif new is None:
                change_type = "REMOVED"
            elif old.get("hash") != new.get("hash"):
                change_type = "CHANGED"
            else:
                continue

            change_rows.append({
                "run_id": RUN_ID,
                "metadata_version": METADATA_VERSION,
                "change_detected_at_utc": extracted_at,
                "workbook_luid": (MANIFEST.get("selected_workbook") or {}).get("id") or CONFIG.get("workbook_luid"),
                "workbook_name": (MANIFEST.get("selected_workbook") or {}).get("name") or CONFIG.get("workbook_name"),
                "table_name": table_name,
                "record_key": identity,
                "change_type": change_type,
                "old_record_hash": old.get("hash") if old else None,
                "new_record_hash": new.get("hash") if new else None,
                "old_record_json": _canonical_json(old.get("record")) if old else None,
                "new_record_json": _canonical_json(new.get("record")) if new else None,
            })

        current_state["tables"][table_name] = current_records

    # Persist immutable per-run snapshot and latest comparison state.
    snapshot_path = snapshots_dir / f"metadata_snapshot_{RUN_ID}.json"
    snapshot_payload = {
        "run_id": RUN_ID,
        "metadata_version": METADATA_VERSION,
        "extracted_at_utc": extracted_at,
        "workbook": current_state["workbook"],
        "tables": {
            table_name: df.to_dict(orient="records") if isinstance(df, pd.DataFrame) else list(df or [])
            for table_name, df in tables.items()
        },
    }
    snapshot_path.write_text(
        json.dumps(snapshot_payload, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )
    state_path.write_text(
        json.dumps(current_state, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )

    columns = [
        "run_id", "metadata_version", "change_detected_at_utc",
        "workbook_luid", "workbook_name", "table_name", "record_key",
        "change_type", "old_record_hash", "new_record_hash",
        "old_record_json", "new_record_json",
    ]
    change_df = pd.DataFrame(change_rows, columns=columns)
    if change_log_path.exists():
        prior = pd.read_csv(change_log_path, dtype=str, keep_default_na=False)
        change_df = pd.concat([prior, change_df], ignore_index=True)
    change_df.to_csv(change_log_path, index=False, encoding="utf-8-sig")

    MANIFEST["metadata_version"] = METADATA_VERSION
    MANIFEST["change_log_path"] = str(change_log_path)
    MANIFEST["snapshot_path"] = str(snapshot_path)
    MANIFEST["metadata_change_counts"] = (
        pd.DataFrame(change_rows)["change_type"].value_counts().to_dict()
        if change_rows else {}
    )
    MANIFEST["metadata_change_rows_this_run"] = len(change_rows)

    # Add run/version stamps to the current extract tables only after diffing,
    # so the timestamps themselves do not trigger false metadata changes.
    if CONFIG.get("add_version_columns_to_outputs", True):
        for table_name, df in list(tables.items()):
            df = df.copy()
            df["run_id"] = RUN_ID
            df["metadata_version"] = METADATA_VERSION
            df["extracted_at_utc"] = extracted_at
            tables[table_name] = df

    print(f"Metadata version: {METADATA_VERSION}")
    print(f"Change-log rows this run: {len(change_rows)}")
    print(f"Change log: {change_log_path}")
    print(f"Immutable snapshot: {snapshot_path}")
    return tables




## Step 10A. TABLEAU VIEW SCREENSHOTS

Capture workbook view screenshots and track screenshot results.

In [ ]:
# ============================================================
# 10A. TABLEAU VIEW SCREENSHOTS
# ============================================================

def capture_workbook_view_screenshots(client, workbook):
    """Export one PNG per workbook view; errors are recorded per view."""
    result = {"status": "SKIPPED", "count": 0, "files": [], "errors": []}
    if not CONFIG.get("capture_view_screenshots", True):
        result["status"] = "DISABLED"
        MANIFEST["screenshots"] = result
        return result
    if client is None or not workbook:
        result["status"] = "SKIPPED"
        result["errors"].append("Screenshots require an authenticated Tableau Cloud/Server client.")
        MANIFEST["screenshots"] = result
        return result

    screenshot_dir = OUTPUT_DIR / CONFIG.get("screenshots_dirname", "view_screenshots")
    screenshot_dir.mkdir(parents=True, exist_ok=True)
    try:
        views = client.get_workbook_views(workbook.get("id"))
        result["views_discovered"] = len(views)
        for view in views:
            view_id = str(view.get("id") or "")
            view_name = str(view.get("name") or view.get("contentUrl") or view_id)
            if not view_id:
                result["errors"].append({
                    "view_name": view_name,
                    "error": "View record did not contain an ID.",
                })
                continue
            filename = safe_filename(view_name, max_length=120) + "_" + view_id[:12] + ".png"
            output_path = screenshot_dir / filename
            try:
                client.download_view_image(view_id, output_path)
                result["files"].append({
                    "view_id": view_id,
                    "view_name": view_name,
                    "path": str(output_path),
                    "size_bytes": output_path.stat().st_size,
                    "sha256": sha256_file(output_path),
                })
            except Exception as exc:
                result["errors"].append({
                    "view_id": view_id,
                    "view_name": view_name,
                    "error": f"{type(exc).__name__}: {exc}",
                })
        result["count"] = len(result["files"])
        result["status"] = (
            "SUCCESS" if not result["errors"] and result.get("count", 0) == result.get("views_discovered", 0)
            else "PARTIAL_SUCCESS"
        )
    except Exception as exc:
        result["status"] = "FAILED"
        result["errors"].append(f"{type(exc).__name__}: {exc}")

    MANIFEST["screenshots"] = result
    print()
    print(f"View screenshots: {result['status']} ({result['count']} saved)")
    print(f"Screenshot directory: {screenshot_dir}")
    if result["errors"]:
        print(f"Screenshot warnings/errors: {len(result['errors'])}")
    return result




## Step 10. OUTPUT WRITER

Write extracted metadata and summaries into the configured output formats.

In [ ]:
# ============================================================
# 10. OUTPUT WRITER
# ============================================================

def write_outputs(
    tables: Dict[str, pd.DataFrame],
    summary: pd.DataFrame,
    qa_df: pd.DataFrame,
):

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    generated_files = []

    # --------------------------------------------------------
    # SUMMARY / QA
    # --------------------------------------------------------

    tables_to_write = dict(tables)

    tables_to_write["summary"] = summary
    tables_to_write["qa"] = qa_df

    # --------------------------------------------------------
    # CSV
    # --------------------------------------------------------

    if CONFIG["export_csv"]:

        csv_dir = OUTPUT_DIR / "csv"

        csv_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        for table_name, df in tables_to_write.items():

            filename = safe_filename(
                table_name
            ) + ".csv"

            path = csv_dir / filename

            df.to_csv(
                path,
                index=False,
                encoding="utf-8-sig",
            )

            generated_files.append(
                str(path)
            )

    # --------------------------------------------------------
    # EXCEL
    # --------------------------------------------------------

    if CONFIG["export_excel"]:

        excel_path = (
            OUTPUT_DIR
            / safe_filename(
                CONFIG["excel_filename"]
            )
        )

        used_sheet_names = set()

        with pd.ExcelWriter(
            excel_path,
            engine="openpyxl",
        ) as writer:

            for table_name, df in tables_to_write.items():

                sheet_name = safe_excel_sheet_name(
                    table_name,
                    used_sheet_names,
                )

                # Ensure Excel can handle an entirely empty
                # dataframe with zero columns.
                if df.empty and len(df.columns) == 0:

                    pd.DataFrame(
                        {
                            "info": [
                                "No records returned."
                            ]
                        }
                    ).to_excel(
                        writer,
                        sheet_name=sheet_name,
                        index=False,
                    )

                else:

                    df.to_excel(
                        writer,
                        sheet_name=sheet_name,
                        index=False,
                    )

        generated_files.append(
            str(excel_path)
        )

    # --------------------------------------------------------
    # JSON
    # --------------------------------------------------------

    if CONFIG["export_json"]:

        json_path = (
            OUTPUT_DIR
            / safe_filename(
                CONFIG["json_filename"]
            )
        )

        json_payload = {
            "generated_utc": utc_now(),
            "config": {
                key: value
                for key, value in CONFIG.items()
                if "secret" not in key.lower()
                and "pat" not in key.lower()
            },
            "tables": {
                table_name: df.to_dict(
                    orient="records"
                )
                for table_name, df
                in tables_to_write.items()
            },
        }

        with open(
            json_path,
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                json_payload,
                f,
                indent=2,
                ensure_ascii=False,
                default=str,
            )

        generated_files.append(
            str(json_path)
        )

    MANIFEST["output_files"] = generated_files
    if MANIFEST.get("change_log_path"):
        generated_files.append(MANIFEST["change_log_path"])
    if MANIFEST.get("snapshot_path"):
        generated_files.append(MANIFEST["snapshot_path"])
    for screenshot in MANIFEST.get("screenshots", {}).get("files", []):
        generated_files.append(screenshot.get("path"))

    print()
    print("Output files written:")

    for file_path in generated_files:
        print(f"  - {file_path}")

    return generated_files




## Step 11. MANIFEST WRITER

Write the execution manifest and run status.

In [ ]:
# ============================================================
# 11. MANIFEST WRITER
# ============================================================

def write_manifest():

    if not CONFIG["write_manifest"]:
        return None

    MANIFEST["run_finished_utc"] = utc_now()

    manifest_path = (
        OUTPUT_DIR
        / "tableau_metadata_run_manifest.json"
    )

    with open(
        manifest_path,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            MANIFEST,
            f,
            indent=2,
            ensure_ascii=False,
            default=str,
        )

    return manifest_path




## Step 12. MAIN PIPELINE

Run the extraction workflow. The original exception handling and final cleanup remain in this same cell to preserve Python control flow.

In [ ]:
# ============================================================
# 12. MAIN PIPELINE
# ============================================================

client = None

selected_workbook = None
resolved_project = None
downloaded_path = None
download_error = None

all_tables = {}

metadata_api_raw = None
twb_raw_info = None

try:

    print("=" * 70)
    print("TABLEAU METADATA EXTRACTION PIPELINE")
    print("=" * 70)

    print()
    print(f"MODE: {CONFIG['mode'].upper()}")

    if CONFIG["mode"].lower() == "cloud":

        print(
            f"Server: {CONFIG['tableau_server']}"
        )

        print(
            f"Site: {CONFIG['site_content_url']}"
        )

        print(
            f"Workbook: {CONFIG['workbook_name']}"
        )

        print(
            f"REST API: {CONFIG['api_version']}"
        )

        print(
            "Project: "
            + " / ".join(
                CONFIG["project_path"]
            )
        )

    else:

        print(
            f"Local workbook: "
            f"{CONFIG['local_workbook']}"
        )

    print()

    # ========================================================
    # CLOUD MODE
    # ========================================================

    if CONFIG["mode"].lower() == "cloud":

        # ----------------------------------------------------
        # PAT
        # ----------------------------------------------------

        pat_name = os.getenv(
            CONFIG["pat_name_env"]
        )

        pat_secret = os.getenv(
            CONFIG["pat_secret_env"]
        )

        if not pat_name:
            raise RuntimeError(
                "Tableau PAT name environment variable is missing.\n\n"
                f"Expected environment variable:\n"
                f"  {CONFIG['pat_name_env']}"
            )

        if not pat_secret:
            raise RuntimeError(
                "Tableau PAT secret environment variable is missing.\n\n"
                f"Expected environment variable:\n"
                f"  {CONFIG['pat_secret_env']}"
            )

        # ----------------------------------------------------
        # CLIENT
        # ----------------------------------------------------

        client = TableauRestClient(
            server=CONFIG["tableau_server"],
            site_content_url=CONFIG["site_content_url"],
            api_version=CONFIG["api_version"],
            pat_name=pat_name,
            pat_secret=pat_secret,
        )

        print("Signing in to Tableau Cloud...")

        client.signin()

        # ----------------------------------------------------
        # RESOLVE WORKBOOK
        # ----------------------------------------------------

        print()
        print("Locating workbook...")

        selected_workbook = client.get_workbook(
            workbook_name=CONFIG["workbook_name"],
            workbook_luid=CONFIG["workbook_luid"],
            project_path=CONFIG["project_path"],
        )

        MANIFEST["selected_workbook"] = {
            "id": selected_workbook.get("id"),
            "name": selected_workbook.get("name"),
            "contentUrl": selected_workbook.get(
                "contentUrl"
            ),
            "project": selected_workbook.get(
                "project"
            ),
            "owner": selected_workbook.get(
                "owner"
            ),
        }

        # ----------------------------------------------------
        # PROJECT INFO
        # ----------------------------------------------------

        project = selected_workbook.get(
            "project",
            {}
        )

        resolved_project = project

        MANIFEST["selected_project"] = {
            "id": project.get("id"),
            "name": project.get("name"),
            "path": CONFIG["project_path"],
        }

        # ----------------------------------------------------
        # SCREENSHOT EVERY WORKBOOK VIEW
        # ----------------------------------------------------

        if CONFIG.get("capture_view_screenshots", True):
            capture_workbook_view_screenshots(
                client=client,
                workbook=selected_workbook,
            )

        # ----------------------------------------------------
        # DOWNLOAD TWBX
        # ----------------------------------------------------

        if CONFIG["download_workbook"]:

            try:
                downloaded_path = client.download_cloud_workbook(
                    selected_workbook
                )

                MANIFEST["downloaded_workbook"] = {
                    "path": str(downloaded_path),
                    "size_bytes": downloaded_path.stat().st_size,
                    "sha256": sha256_file(downloaded_path),
                }

            except Exception as exc:
                # Metadata API extraction is independent of TWB/TWBX download.
                # Keep going and mark the run as partial instead of losing API metadata.
                download_error = f"{type(exc).__name__}: {exc}"
                MANIFEST["errors"].append(
                    f"WorkbookDownloadWarning: {download_error}"
                )
                print()
                print("WARNING: Workbook file download failed.")
                print(download_error)
                print(
                    "Continuing with Metadata API extraction; "
                    "TWB/TWBX-only details will be unavailable."
                )

        # ----------------------------------------------------
        # METADATA API
        # ----------------------------------------------------

        if (
            CONFIG["query_metadata_api"]
            and CONFIG["parse_metadata_records"]
        ):

            workbook_luid = selected_workbook.get(
                "id"
            )

            metadata_api_raw = (
                client.query_workbook_metadata(
                    workbook_luid
                )
            )

            metadata_tables = (
                normalize_metadata_api(
                    metadata_api_raw
                )
            )

            for table_name, records in metadata_tables.items():

                all_tables[table_name] = (
                    dataframe_from_records(
                        records
                    )
                )

        # ----------------------------------------------------
        # TWB/TWBX PARSING
        # ----------------------------------------------------

        if (
            CONFIG["parse_twb"]
            and downloaded_path is not None
        ):

            print()
            print("Parsing downloaded Tableau workbook...")

            (
                source_path,
                xml_bytes,
                tree,
                twb_raw_info,
            ) = load_twb_from_path(
                downloaded_path
            )

            extractor = TWBExtractor(
                tree=tree,
                source_info=twb_raw_info,
            )

            twb_tables = extractor.run()

            for table_name, records in twb_tables.items():

                all_tables[
                    f"twb_{table_name}"
                ] = dataframe_from_records(
                    records
                )

            print("TWB/TWBX parsing successful.")

    # ========================================================
    # LOCAL MODE
    # ========================================================

    elif CONFIG["mode"].lower() == "local":

        local_path = Path(
            CONFIG["local_workbook"]
        )

        print()
        print("Loading local Tableau workbook...")
        print(f"  Path: {local_path}")

        (
            source_path,
            xml_bytes,
            tree,
            twb_raw_info,
        ) = load_twb_from_path(
            local_path
        )

        extractor = TWBExtractor(
            tree=tree,
            source_info=twb_raw_info,
        )

        twb_tables = extractor.run()

        for table_name, records in twb_tables.items():

            all_tables[
                table_name
            ] = dataframe_from_records(
                records
            )

        print("Local TWB/TWBX parsing successful.")

    # ========================================================
    # DATAFRAME NORMALIZATION
    # ========================================================

    for table_name in list(all_tables.keys()):

        df = all_tables[table_name]

        if not isinstance(df, pd.DataFrame):
            df = dataframe_from_records(df)

        # Normalize object columns to strings where useful,
        # while preserving numeric/date types.
        for col in df.columns:

            if df[col].dtype == "object":

                try:
                    df[col] = df[col].where(
                        df[col].notna(),
                        None,
                    )
                except Exception:
                    pass

        all_tables[table_name] = df

    # ========================================================
    # METADATA VERSIONING / CHANGE HISTORY
    # ========================================================

    all_tables = track_metadata_history(all_tables)

    # ========================================================
    # SUMMARY
    # ========================================================

    summary = build_summary(
        all_tables
    )

    # ========================================================
    # QA
    # ========================================================

    qa_df = build_qa(
        tables=all_tables,
        selected_workbook=selected_workbook,
        selected_project=resolved_project,
    )

    # ========================================================
    # PRINT SUMMARY
    # ========================================================

    print()
    print("=" * 70)
    print("EXTRACTION SUMMARY")
    print("=" * 70)

    display(summary)

    print()
    print("=" * 70)
    print("QUALITY ASSURANCE")
    print("=" * 70)

    display(qa_df)

    # ========================================================
    # FAIL IF QA HAS HARD FAILURES
    # ========================================================

    hard_failures = qa_df[
        qa_df["status"].astype(str).str.upper() == "FAIL"
    ]

    if not hard_failures.empty:

        raise RuntimeError(
            "Extraction completed, but QA detected hard failures:\n\n"
            + hard_failures.to_string(index=False)
        )

    # ========================================================
    # WRITE OUTPUTS
    # ========================================================

    write_outputs(
        tables=all_tables,
        summary=summary,
        qa_df=qa_df,
    )

    screenshot_status = MANIFEST.get("screenshots", {}).get("status")
    MANIFEST["status"] = (
        "PARTIAL_SUCCESS"
        if download_error or screenshot_status in ("PARTIAL_SUCCESS", "FAILED")
        else "SUCCESS"
    )

    print()
    print("=" * 70)
    if MANIFEST["status"] == "PARTIAL_SUCCESS":
        print("METADATA EXTRACTION COMPLETED WITH WARNINGS")
        if download_error:
            print("Workbook download failed; TWB/TWBX parsing may be unavailable.")
        if screenshot_status in ("PARTIAL_SUCCESS", "FAILED"):
            print("Some or all view screenshots could not be captured; see manifest.")
    else:
        print("EXTRACTION COMPLETED SUCCESSFULLY")
    print("=" * 70)

# ============================================================
# ERROR HANDLING
# ============================================================

except Exception as exc:

    MANIFEST["status"] = "FAILED"

    error_text = (
        f"{type(exc).__name__}: {exc}"
    )

    MANIFEST["errors"].append(
        error_text
    )

    print()
    print("=" * 70)
    print("EXTRACTION FAILED")
    print("=" * 70)
    print()
    print(error_text)

    print()
    print("Traceback:")
    traceback.print_exc()

    # Preserve the original exception.
    #
    # Sign-out is handled separately in finally and will NEVER
    # replace this exception.

    raise

# ============================================================
# ALWAYS CLEAN UP
# ============================================================

finally:

    # --------------------------------------------------------
    # Sign out WITHOUT masking original exception.
    # --------------------------------------------------------

    if client is not None:

        try:

            client.signout()

        except Exception as signout_exc:

            print()
            print(
                "WARNING: Tableau sign-out failed, "
                "but the original extraction result/error "
                "has been preserved."
            )

            print(
                f"Sign-out error: {signout_exc}"
            )

            MANIFEST["errors"].append(
                f"SignoutError: {signout_exc}"
            )

    # --------------------------------------------------------
    # Always finalize manifest.
    # --------------------------------------------------------

    try:

        manifest_path = write_manifest()

        if manifest_path:

            print()
            print(
                f"Run manifest: {manifest_path}"
            )

    except Exception as manifest_exc:

        print()
        print(
            "WARNING: Could not write run manifest:"
        )

        print(
            f"  {manifest_exc}"
        )